<a href="https://colab.research.google.com/github/SAPalm2024/IITK/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re

##### You may comment this section to see verbose -- but you must un-comment this before final submission. ######
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
#################################################################################################################

"""
* * * Changes allowed from here  * * *
"""


def llm_function(model, tokenizer, questions):
    '''
    The steps are given for your reference:

    1. Generate answer for the first question.
    2. Generate answer for the second question use the answer for first question as context.
    3. Generate a deterministic output either 'YES' or 'NO' for the third question using the context from second question.
    5. Clean output and return.
    6. Output is case-sensative: YES or NO
    Note: The model (Flan-T5-XL) and tokenizer is already initialized. Do not modify that section.
    '''

    def generate(prompt, max_new_tokens=150):
        """Greedy decode — do_sample=False is the default, so generation is deterministic."""
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )
        return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

    # ── Step 1: Generate answer to Q1 ───────────────────────
    a1 = generate(f"Q: {questions[0]}\nA:")

    # ── Step 2: Generate answer to Q2, using Q1→A1 as one-shot context ───
    a2 = generate(f"Q: {questions[0]}\nA: {a1}\nQ: {questions[1]}\nA:")

    # ── Step 3: YES/NO for Q3, using Q2→A2 as one-shot context ────────
    # "Please answer yes or no." nudges Flan-T5-XL toward a single-word output.
    prompt3 = (
        f"Q: {questions[1]}\nA: {a2}\n"
        f"Q: {questions[2]} Please answer yes or no.\nA:"
    )
    raw3 = generate(prompt3, max_new_tokens=10)

    # Primary extraction: word-boundary regex, case-insensitive
    match = re.search(r'\b(yes|no)\b', raw3, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    # ── Fallback: compare first-token logits for 'yes' vs 'no' ─────────
    # Checks lowercase, title-case, and uppercase surface forms because the
    # SentencePiece tokeniser assigns different IDs to each.  We guard with
    # len(ids) == 1 so a multi-token encoding never silently picks the wrong ID.
    input_ids_f = tokenizer(prompt3, return_tensors="pt").input_ids
    decoder_start = torch.tensor([[model.config.decoder_start_token_id]])

    with torch.no_grad():
        logits = model(
            input_ids=input_ids_f,
            decoder_input_ids=decoder_start,
        ).logits[0, 0]   # shape: (vocab_size,)

    scores = {"YES": float("-inf"), "NO": float("-inf")}
    for surface, label in [
        ("yes", "YES"), ("Yes", "YES"), ("YES", "YES"),
        ("no",  "NO"),  ("No",  "NO"),  ("NO",  "NO"),
    ]:
        ids = tokenizer(surface, add_special_tokens=False).input_ids
        if len(ids) == 1:
            scores[label] = max(scores[label], logits[ids[0]].item())

    final_output = max(scores, key=scores.get)
    return final_output


"""
ALERT: * * * No changes are allowed below this comment  * * *
"""

if __name__ == '__main__':

    # To make the code runnable in Colab, we'll use hardcoded questions
    # instead of sys.argv. You can change these questions to test different scenarios.
    question_a = "Who is Rabindranath Tagore?"
    question_b = "Where was he born?"
    question_c = "Is it in India?"

    questions = [question_a, question_b, question_c]
    ##################### Loading Model and Tokenizer ########################
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl")
    model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl")
    ##########################################################################

    """  Call to function that will perform the computation. """
    torch.manual_seed(42)
    out = llm_function(model,tokenizer,questions)
    print(out.strip())

    """ End to call """


# New Section